In [ ]:
import os
import math
import queue
import threading
import time
import urllib.request
from pathlib import Path
from collections import deque

import numpy as np
import soundcard as sc
from netCDF4 import Dataset

import tkinter as tk
from tkinter import ttk, messagebox, filedialog


# ============================================================
# Audio Radar V4
# SOFA HRTF template matching + Windows WASAPI loopback
#
# 角度顯示：
#   0°    = 前
#   +90°  = 右
#   -90°  = 左
#   ±180° = 後
#
# 注意：
# 這不是神經網路。它利用 SOFA HRTF 的左右耳頻譜比例、
# Interaural Phase Difference、Level Difference 去做 360° 模板比對。
# ============================================================

CAPTURE_SR = 48000
CHUNK = 4096
MIN_RMS = 0.0025
SMOOTHING = 0.72

# 比對頻率範圍
FREQ_MIN = 300.0
FREQ_MAX = 12000.0
N_FEATURE_FREQS = 120

# 官方 SOFA repository 的 MIT KEMAR normal pinna
MIT_KEMAR_URL = (
    "https://sofacoustics.org/data/database/mit/"
    "mit_kemar_normal_pinna.sofa"
)

APP_DIR = Path.home() / ".audio_radar"
DEFAULT_SOFA = APP_DIR / "mit_kemar_normal_pinna.sofa"


def rms(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        return 0.0
    return float(np.sqrt(np.mean(x * x) + 1e-12))


def wrap180(angle):
    return ((float(angle) + 180.0) % 360.0) - 180.0


def circular_smooth(old_deg, new_deg, amount=SMOOTHING):
    old_r = math.radians(old_deg)
    new_r = math.radians(new_deg)

    x = amount * math.cos(old_r) + (1.0 - amount) * math.cos(new_r)
    y = amount * math.sin(old_r) + (1.0 - amount) * math.sin(new_r)

    return math.degrees(math.atan2(y, x))


def next_pow2(n):
    n = max(1, int(n))
    return 1 << (n - 1).bit_length()


def interp_real(freq_src, values, freq_dst):
    return np.interp(freq_dst, freq_src, values)


def interp_unit_complex(freq_src, z, freq_dst):
    """
    分別插值 unit complex 的 real/imag，再重新正規化。
    避免直接對 wrapped phase 做線性插值。
    """
    real = np.interp(freq_dst, freq_src, np.real(z))
    imag = np.interp(freq_dst, freq_src, np.imag(z))
    out = real + 1j * imag
    out /= np.abs(out) + 1e-12
    return out


def source_positions_to_spherical(source_pos, pos_type):
    """
    回傳 azimuth, elevation, distance。
    SOFA spherical 預設為 degree, degree, metre。
    cartesian 則依 x=front, y=left, z=up 轉換。
    """
    p = np.asarray(source_pos, dtype=np.float64)

    while p.ndim > 2:
        p = p[:, 0, :]

    if p.ndim != 2 or p.shape[1] < 3:
        raise RuntimeError(
            f"SourcePosition 維度無法處理：shape={p.shape}"
        )

    t = (pos_type or "spherical").lower()

    if "spherical" in t:
        az = p[:, 0]
        el = p[:, 1]
        dist = p[:, 2]
        return az, el, dist

    if "cartesian" in t:
        x = p[:, 0]
        y = p[:, 1]
        z = p[:, 2]

        az = np.degrees(np.arctan2(y, x))
        horizontal = np.sqrt(x * x + y * y)
        el = np.degrees(np.arctan2(z, horizontal))
        dist = np.sqrt(x * x + y * y + z * z)
        return az, el, dist

    raise RuntimeError(
        f"不支援 SourcePosition Type：{pos_type}"
    )


def receiver_left_right_indices(ds, n_receivers):
    """
    SOFA 預設座標：x=front, y=left, z=up。
    因此 ReceiverPosition y > 0 為左耳，y < 0 為右耳。

    若檔案沒有足夠資訊，退回 receiver 0=left, 1=right。
    """
    if n_receivers < 2:
        raise RuntimeError("SOFA 檔只有一個 receiver，無法做 binaural 比對。")

    try:
        rp_var = ds.variables["ReceiverPosition"]
        rp = np.asarray(rp_var[:], dtype=np.float64)

        # 常見 shape: [R, C, I]、[R, C] 或 [I, R, C]
        rp = np.squeeze(rp)

        candidates = []

        if rp.ndim == 2:
            if rp.shape[0] == n_receivers and rp.shape[1] >= 3:
                candidates = rp[:, :3]
            elif rp.shape[1] == n_receivers and rp.shape[0] >= 3:
                candidates = rp[:3, :].T

        elif rp.ndim == 3:
            # 找包含 receiver 軸的合理切片
            for axis in range(3):
                if rp.shape[axis] == n_receivers:
                    moved = np.moveaxis(rp, axis, 0)
                    flat = moved.reshape(n_receivers, -1)
                    if flat.shape[1] >= 3:
                        candidates = flat[:, :3]
                        break

        if len(candidates) >= 2:
            y = np.asarray(candidates)[:, 1]
            left = int(np.argmax(y))
            right = int(np.argmin(y))
            if left != right:
                return left, right

    except Exception:
        pass

    return 0, 1


class HRTF360Matcher:
    def __init__(self):
        self.loaded = False
        self.path = None

        self.feature_freqs = None
        self.templates = []
        self.sample_rate = None
        self.elevation_used = None

        self.history_ild = deque(maxlen=3)
        self.history_ipd = deque(maxlen=3)
        self.history_weight = deque(maxlen=3)

    def reset_history(self):
        self.history_ild.clear()
        self.history_ipd.clear()
        self.history_weight.clear()

    def load_sofa(self, path):
        path = str(path)

        with Dataset(path, "r") as ds:
            if "Data.IR" not in ds.variables:
                raise RuntimeError("這個 SOFA 檔沒有 Data.IR。")

            if "SourcePosition" not in ds.variables:
                raise RuntimeError("這個 SOFA 檔沒有 SourcePosition。")

            ir = np.asarray(ds.variables["Data.IR"][:], dtype=np.float64)
            source_var = ds.variables["SourcePosition"]
            source_pos = np.asarray(source_var[:], dtype=np.float64)

            pos_type = getattr(source_var, "Type", "spherical")

            if "Data.SamplingRate" in ds.variables:
                sr_arr = np.asarray(
                    ds.variables["Data.SamplingRate"][:],
                    dtype=np.float64
                ).reshape(-1)
                hrtf_sr = float(sr_arr[0])
            else:
                hrtf_sr = 48000.0

            # Data.IR 一般是 [M, R, N]
            ir = np.squeeze(ir)

            if ir.ndim != 3:
                raise RuntimeError(
                    f"目前程式預期 Data.IR 為三維 [M,R,N]，"
                    f"實際 shape={ir.shape}"
                )

            m, r, n = ir.shape

            if r < 2:
                raise RuntimeError("Data.IR receiver 數量不足 2。")

            az, el, dist = source_positions_to_spherical(
                source_pos, pos_type
            )

            if len(az) != m:
                raise RuntimeError(
                    f"SourcePosition 數量 {len(az)} "
                    f"與 Data.IR measurement 數量 {m} 不一致。"
                )

            left_idx, right_idx = receiver_left_right_indices(ds, r)

            # 找最接近水平面的 elevation。
            min_abs_el = float(np.min(np.abs(el)))
            horizontal_mask = np.abs(el) <= max(2.0, min_abs_el + 0.5)

            indices = np.where(horizontal_mask)[0]

            if len(indices) < 8:
                # 再放寬
                horizontal_mask = np.abs(el) <= max(10.0, min_abs_el + 2.0)
                indices = np.where(horizontal_mask)[0]

            if len(indices) < 8:
                raise RuntimeError(
                    "SOFA 檔在水平面附近的測量方向太少，"
                    "不適合做 360° 水平定位。"
                )

            self.elevation_used = float(np.median(el[indices]))

            max_f = min(FREQ_MAX, hrtf_sr * 0.48)
            self.feature_freqs = np.geomspace(
                FREQ_MIN,
                max_f,
                N_FEATURE_FREQS
            )

            nfft = next_pow2(max(n, 2048))
            fft_freqs = np.fft.rfftfreq(nfft, d=1.0 / hrtf_sr)

            templates = []

            # 有些 SOFA 水平面會出現重複 azimuth，最後合併
            by_angle = {}

            for idx in indices:
                h_l = ir[idx, left_idx, :]
                h_r = ir[idx, right_idx, :]

                H_l = np.fft.rfft(h_l, n=nfft)
                H_r = np.fft.rfft(h_r, n=nfft)

                mag_l = np.abs(H_l) + 1e-12
                mag_r = np.abs(H_r) + 1e-12

                ild_full = 20.0 * np.log10(mag_r / mag_l)
                ild = interp_real(
                    fft_freqs,
                    ild_full,
                    self.feature_freqs
                )
                ild = np.clip(ild, -35.0, 35.0)

                cross_unit = H_r * np.conj(H_l)
                cross_unit /= np.abs(cross_unit) + 1e-12

                ipd = interp_unit_complex(
                    fft_freqs,
                    cross_unit,
                    self.feature_freqs
                )

                # SOFA: +azimuth 通常是朝左。
                # 本 GUI 希望 +90° 是右，因此符號反轉。
                display_angle = wrap180(-az[idx])

                # 接近同角度的 measurement 合併
                key = round(display_angle, 2)

                item = {
                    "angle": display_angle,
                    "ild": ild,
                    "ipd": ipd,
                }

                by_angle.setdefault(key, []).append(item)

            for key, items in by_angle.items():
                ild_stack = np.stack([x["ild"] for x in items], axis=0)
                ipd_stack = np.stack([x["ipd"] for x in items], axis=0)

                ild_avg = np.mean(ild_stack, axis=0)
                ipd_avg = np.mean(ipd_stack, axis=0)
                ipd_avg /= np.abs(ipd_avg) + 1e-12

                templates.append({
                    "angle": float(np.mean([x["angle"] for x in items])),
                    "ild": ild_avg,
                    "ipd": ipd_avg,
                })

            templates.sort(key=lambda x: x["angle"])

        self.templates = templates
        self.sample_rate = hrtf_sr
        self.path = path
        self.loaded = True
        self.reset_history()

        return {
            "directions": len(self.templates),
            "sample_rate": self.sample_rate,
            "elevation": self.elevation_used,
        }

    def extract_observation(self, stereo):
        stereo = np.asarray(stereo, dtype=np.float64)

        if stereo.ndim != 2 or stereo.shape[1] < 2:
            return None

        left = stereo[:, 0] - np.mean(stereo[:, 0])
        right = stereo[:, 1] - np.mean(stereo[:, 1])

        strength = max(rms(left), rms(right))

        if strength < MIN_RMS:
            return None

        n = len(left)
        nfft = next_pow2(max(n, 4096))

        win = np.hanning(n)
        L = np.fft.rfft(left * win, n=nfft)
        R = np.fft.rfft(right * win, n=nfft)

        freqs = np.fft.rfftfreq(nfft, d=1.0 / CAPTURE_SR)

        mag_l = np.abs(L) + 1e-12
        mag_r = np.abs(R) + 1e-12

        ild_full = 20.0 * np.log10(mag_r / mag_l)
        ild = interp_real(
            freqs,
            ild_full,
            self.feature_freqs
        )
        ild = np.clip(ild, -35.0, 35.0)

        cross = R * np.conj(L)
        cross_unit = cross / (np.abs(cross) + 1e-12)

        ipd = interp_unit_complex(
            freqs,
            cross_unit,
            self.feature_freqs
        )

        # 頻率權重：該頻率真的有能量才相信它
        energy_full = np.sqrt(mag_l * mag_r)
        energy = interp_real(
            freqs,
            energy_full,
            self.feature_freqs
        )

        # log 壓縮，避免單一頻率完全支配
        weight = np.log1p(
            energy / (np.median(energy) + 1e-12)
        )
        weight = np.clip(weight, 0.05, None)
        weight /= np.sum(weight) + 1e-12

        self.history_ild.append(ild)
        self.history_ipd.append(ipd)
        self.history_weight.append(weight)

        ild_avg = np.median(
            np.stack(self.history_ild, axis=0),
            axis=0
        )

        ipd_avg = np.mean(
            np.stack(self.history_ipd, axis=0),
            axis=0
        )
        ipd_avg /= np.abs(ipd_avg) + 1e-12

        weight_avg = np.mean(
            np.stack(self.history_weight, axis=0),
            axis=0
        )
        weight_avg /= np.sum(weight_avg) + 1e-12

        return {
            "ild": ild_avg,
            "ipd": ipd_avg,
            "weight": weight_avg,
            "strength": strength,
        }

    def score_template(self, obs, template):
        freqs = self.feature_freqs
        w = obs["weight"]

        low_mask = freqs <= 2200.0
        high_mask = freqs >= 1800.0

        # ---- 低頻 IPD：主要提供左右/ITD cue ----
        wl = w * low_mask
        wl /= np.sum(wl) + 1e-12

        phase_cos_low = np.real(
            obs["ipd"] * np.conj(template["ipd"])
        )
        phase_score_low = float(np.sum(wl * phase_cos_low))
        phase_score_low = 0.5 * (phase_score_low + 1.0)

        # ---- 高頻 ILD/頻譜形狀：前後 discrimination 關鍵 ----
        wh = w * high_mask
        wh /= np.sum(wh) + 1e-12

        o = obs["ild"].copy()
        t = template["ild"].copy()

        # 去除固定左右增益偏移，只比較 frequency-dependent shape
        o_shape = o - np.sum(wh * o)
        t_shape = t - np.sum(wh * t)

        num = np.sum(wh * o_shape * t_shape)
        den = (
            math.sqrt(np.sum(wh * o_shape * o_shape) + 1e-12)
            * math.sqrt(np.sum(wh * t_shape * t_shape) + 1e-12)
        )

        ild_shape = float(num / (den + 1e-12))
        ild_shape_score = 0.5 * (ild_shape + 1.0)

        # ---- 整體 ILD：保留左右哪一耳較強的資訊 ----
        mid_mask = (freqs >= 500.0) & (freqs <= 5000.0)
        wm = w * mid_mask
        wm /= np.sum(wm) + 1e-12

        obs_mean_ild = float(np.sum(wm * obs["ild"]))
        tpl_mean_ild = float(np.sum(wm * template["ild"]))

        level_error = abs(obs_mean_ild - tpl_mean_ild)
        level_score = math.exp(-level_error / 9.0)

        # ---- 中高頻相位只給小權重 ----
        phase_cos_all = np.real(
            obs["ipd"] * np.conj(template["ipd"])
        )
        phase_score_all = float(np.sum(w * phase_cos_all))
        phase_score_all = 0.5 * (phase_score_all + 1.0)

        # 總分 0~1（大致）
        score = (
            0.42 * phase_score_low
            + 0.38 * ild_shape_score
            + 0.15 * level_score
            + 0.05 * phase_score_all
        )

        return float(score)

    def estimate(self, stereo):
        if not self.loaded:
            return None

        obs = self.extract_observation(stereo)

        if obs is None:
            return {
                "active": False,
                "angle": 0.0,
                "confidence": 0.0,
                "strength": 0.0,
                "top": [],
            }

        scored = []

        for tpl in self.templates:
            s = self.score_template(obs, tpl)
            scored.append((s, tpl["angle"]))

        scored.sort(reverse=True, key=lambda x: x[0])

        best_score, best_angle = scored[0]

        # 第二候選不要選相鄰 5° 那種，至少離最佳方向 25°
        second_score = 0.0
        second_angle = best_angle

        for s, a in scored[1:]:
            diff = abs(wrap180(a - best_angle))
            if diff >= 25.0:
                second_score = s
                second_angle = a
                break

        margin = max(0.0, best_score - second_score)

        # confidence 不是機率，只是「最佳 vs 其他候選」的分離程度
        confidence = (
            45.0 * np.clip((best_score - 0.45) / 0.35, 0.0, 1.0)
            + 55.0 * np.clip(margin / 0.20, 0.0, 1.0)
        )

        top = [
            {
                "score": float(s),
                "angle": float(a),
            }
            for s, a in scored[:5]
        ]

        return {
            "active": True,
            "angle": float(best_angle),
            "confidence": float(np.clip(confidence, 0.0, 100.0)),
            "strength": float(obs["strength"]),
            "best_score": float(best_score),
            "second_score": float(second_score),
            "second_angle": float(second_angle),
            "top": top,
        }


class AudioRadarV4:
    def __init__(self, root):
        self.root = root
        self.root.title("Audio Radar V4.1 - HRTF 360° + Relative Distance")
        self.root.geometry("1120x790")
        self.root.minsize(980, 690)

        self.running = False
        self.worker = None
        self.result_queue = queue.Queue(maxsize=4)

        self.loopbacks = []
        self.matcher = HRTF360Matcher()

        self.angle_smoothed = 0.0
        self.current_result = None

        self.device_var = tk.StringVar()
        self.status_var = tk.StringVar(value="尚未開始")
        self.hrtf_var = tk.StringVar(value="HRTF 尚未載入")

        self.angle_var = tk.StringVar(value="--")
        self.conf_var = tk.StringVar(value="--")
        self.score_var = tk.StringVar(value="--")
        self.second_var = tk.StringVar(value="--")
        self.strength_var = tk.StringVar(value="--")
        self.distance_var = tk.StringVar(value="--")
        self.distance_pct_var = tk.StringVar(value="--")
        self.near_ref_var = tk.DoubleVar(value=0.080)
        self.far_ref_var = tk.DoubleVar(value=0.006)
        self.distance_smoothed = 0.5
        self.top_var = tk.StringVar(value="尚無結果")

        self.build_ui()
        self.refresh_devices()

        self.root.after(30, self.process_results)
        self.root.protocol("WM_DELETE_WINDOW", self.on_close)

        # 如果以前已下載過，直接載入
        if DEFAULT_SOFA.exists():
            threading.Thread(
                target=self.load_sofa_worker,
                args=(str(DEFAULT_SOFA),),
                daemon=True
            ).start()
        else:
            self.hrtf_var.set("按「下載 MIT-KEMAR」")

    def build_ui(self):
        top = ttk.Frame(self.root, padding=10)
        top.pack(fill="x")

        ttk.Label(top, text="輸出裝置：").pack(side="left")

        self.device_combo = ttk.Combobox(
            top,
            textvariable=self.device_var,
            state="readonly",
            width=55
        )
        self.device_combo.pack(side="left", padx=5)

        ttk.Button(
            top, text="重新整理", command=self.refresh_devices
        ).pack(side="left", padx=3)

        ttk.Button(
            top, text="開始", command=self.start
        ).pack(side="left", padx=3)

        ttk.Button(
            top, text="停止", command=self.stop
        ).pack(side="left", padx=3)

        hrtf_bar = ttk.Frame(self.root, padding=(10, 0, 10, 10))
        hrtf_bar.pack(fill="x")

        ttk.Label(
            hrtf_bar,
            textvariable=self.hrtf_var
        ).pack(side="left")

        ttk.Button(
            hrtf_bar,
            text="下載 MIT-KEMAR",
            command=self.download_kemar
        ).pack(side="right", padx=3)

        ttk.Button(
            hrtf_bar,
            text="選擇其他 SOFA",
            command=self.choose_sofa
        ).pack(side="right", padx=3)

        main = ttk.Frame(self.root, padding=(10, 0, 10, 10))
        main.pack(fill="both", expand=True)

        self.canvas = tk.Canvas(
            main,
            width=735,
            height=640,
            bg="#111111",
            highlightthickness=0
        )
        self.canvas.pack(side="left", fill="both", expand=True)

        side = ttk.Frame(main)
        side.pack(side="right", fill="y", padx=(12, 0))

        info = ttk.LabelFrame(side, text="360° 定位", padding=10)
        info.pack(fill="x", pady=(0, 8))

        rows = [
            ("狀態", self.status_var),
            ("方向", self.angle_var),
            ("信心度", self.conf_var),
            ("最佳比對", self.score_var),
            ("第二候選", self.second_var),
            ("聲音強度", self.strength_var),
            ("相對距離", self.distance_var),
            ("距離比例", self.distance_pct_var),
        ]

        for i, (name, var) in enumerate(rows):
            ttk.Label(
                info, text=name + "："
            ).grid(row=i, column=0, sticky="w", pady=4)

            ttk.Label(
                info, textvariable=var, width=20
            ).grid(row=i, column=1, sticky="w", pady=4)

        dist_box = ttk.LabelFrame(side, text="距離校準", padding=10)
        dist_box.pack(fill="x", pady=(0, 8))

        ttk.Label(dist_box, text="近距離 RMS：").grid(row=0, column=0, sticky="w")
        ttk.Scale(
            dist_box,
            from_=0.015,
            to=0.20,
            variable=self.near_ref_var,
            orient="horizontal",
            length=150
        ).grid(row=0, column=1, padx=4)

        ttk.Label(dist_box, text="遠距離 RMS：").grid(row=1, column=0, sticky="w")
        ttk.Scale(
            dist_box,
            from_=0.001,
            to=0.03,
            variable=self.far_ref_var,
            orient="horizontal",
            length=150
        ).grid(row=1, column=1, padx=4)

        self.near_ref_label = ttk.Label(dist_box, width=7)
        self.near_ref_label.grid(row=0, column=2)
        self.far_ref_label = ttk.Label(dist_box, width=7)
        self.far_ref_label.grid(row=1, column=2)

        ttk.Label(
            dist_box,
            text="近 = 點靠中心；遠 = 點靠外圈",
            justify="left"
        ).grid(row=2, column=0, columnspan=3, sticky="w", pady=(6, 0))

        top_box = ttk.LabelFrame(side, text="Top HRTF 候選", padding=10)
        top_box.pack(fill="x")

        ttk.Label(
            top_box,
            textvariable=self.top_var,
            justify="left",
            width=27
        ).pack(anchor="w")

        ttk.Label(
            side,
            text=(
                "\n角度定義\n"
                "0° = 前\n"
                "+90° = 右\n"
                "-90° = 左\n"
                "±180° = 後\n\n"
                "前/後主要依賴高頻 HRTF\n"
                "頻譜形狀，因此遊戲必須真的\n"
                "輸出 binaural / HRTF cues。\n\n"
                "建議一次只開一層空間音效。"
            ),
            justify="left"
        ).pack(anchor="w")

    def set_hrtf_text(self, text):
        self.root.after(0, lambda: self.hrtf_var.set(text))

    def choose_sofa(self):
        path = filedialog.askopenfilename(
            title="選擇 SOFA HRTF",
            filetypes=[
                ("SOFA HRTF", "*.sofa"),
                ("All files", "*.*")
            ]
        )

        if not path:
            return

        threading.Thread(
            target=self.load_sofa_worker,
            args=(path,),
            daemon=True
        ).start()

    def download_kemar(self):
        threading.Thread(
            target=self.download_kemar_worker,
            daemon=True
        ).start()

    def download_kemar_worker(self):
        try:
            APP_DIR.mkdir(parents=True, exist_ok=True)

            self.set_hrtf_text("正在下載 MIT-KEMAR HRTF...")

            req = urllib.request.Request(
                MIT_KEMAR_URL,
                headers={"User-Agent": "Mozilla/5.0"}
            )

            with urllib.request.urlopen(req, timeout=45) as response:
                data = response.read()

            if len(data) < 100000:
                raise RuntimeError(
                    "下載的檔案大小異常，可能不是有效 SOFA。"
                )

            DEFAULT_SOFA.write_bytes(data)

            self.load_sofa_worker(str(DEFAULT_SOFA))

        except Exception as e:
            self.set_hrtf_text("MIT-KEMAR 下載失敗")
            self.root.after(
                0,
                lambda: messagebox.showerror(
                    "HRTF 下載失敗",
                    f"{e}\n\n"
                    "可以改按「選擇其他 SOFA」手動載入。"
                )
            )

    def load_sofa_worker(self, path):
        try:
            self.set_hrtf_text("正在建立 360° HRTF 模板...")

            info = self.matcher.load_sofa(path)

            name = Path(path).name

            text = (
                f"HRTF：{name} | "
                f"{info['directions']} directions | "
                f"{info['sample_rate']:.0f} Hz | "
                f"elev {info['elevation']:+.1f}°"
            )

            self.set_hrtf_text(text)

        except Exception as e:
            self.set_hrtf_text("HRTF 載入失敗")
            self.root.after(
                0,
                lambda: messagebox.showerror(
                    "SOFA 載入錯誤",
                    str(e)
                )
            )

    def refresh_devices(self):
        if self.running:
            return

        try:
            default_sp = sc.default_speaker()
            speakers = sc.all_speakers()
            mics = sc.all_microphones(include_loopback=True)

            pairs = []
            used = set()

            for sp in speakers:
                best = None

                for mic in mics:
                    if (
                        getattr(mic, "id", None)
                        == getattr(sp, "id", None)
                    ):
                        best = mic
                        break

                if best is None:
                    s = sp.name.lower()

                    for mic in mics:
                        m = mic.name.lower()

                        if s in m or m in s:
                            best = mic
                            break

                if best is not None:
                    key = str(getattr(best, "id", best.name))

                    if key not in used:
                        used.add(key)
                        pairs.append((sp, best))

            if not pairs:
                pairs = [(None, mic) for mic in mics]

            self.loopbacks = pairs

            names = []
            default_index = 0

            for i, (sp, mic) in enumerate(pairs):
                name = sp.name if sp is not None else mic.name
                names.append(f"{i}: {name}")

                if (
                    sp is not None
                    and default_sp is not None
                    and sp.name == default_sp.name
                ):
                    default_index = i

            self.device_combo["values"] = names

            if names:
                self.device_combo.current(default_index)
                self.status_var.set("裝置已載入")
            else:
                self.status_var.set("找不到 Loopback")

        except Exception as e:
            messagebox.showerror("裝置錯誤", str(e))

    def get_selected_mic(self):
        idx = self.device_combo.current()

        if idx < 0 or idx >= len(self.loopbacks):
            return None

        return self.loopbacks[idx][1]

    def start(self):
        if self.running:
            return

        if not self.matcher.loaded:
            messagebox.showinfo(
                "提示",
                "請先下載 MIT-KEMAR，或選擇一個 SOFA HRTF 檔。"
            )
            return

        mic = self.get_selected_mic()

        if mic is None:
            messagebox.showwarning(
                "提示",
                "請先選擇輸出裝置。"
            )
            return

        self.matcher.reset_history()
        self.angle_smoothed = 0.0
        self.distance_smoothed = 0.5

        self.running = True
        self.status_var.set("正在做 HRTF 360° 比對")

        self.worker = threading.Thread(
            target=self.audio_loop,
            args=(mic,),
            daemon=True
        )
        self.worker.start()

    def stop(self):
        self.running = False
        self.status_var.set("已停止")

    def audio_loop(self, mic):
        try:
            with mic.recorder(
                samplerate=CAPTURE_SR,
                channels=[0, 1],
                blocksize=8192
            ) as recorder:

                while self.running:
                    data = recorder.record(numframes=CHUNK)

                    if data is None or len(data) == 0:
                        continue

                    data = np.asarray(data, dtype=np.float32)

                    if data.ndim != 2 or data.shape[1] < 2:
                        raise RuntimeError(
                            f"沒有取得 stereo，shape={data.shape}"
                        )

                    stereo = data[:, :2]

                    result = self.matcher.estimate(stereo)

                    if result is None:
                        continue

                    try:
                        while self.result_queue.qsize() >= 3:
                            self.result_queue.get_nowait()

                        self.result_queue.put_nowait(result)

                    except queue.Full:
                        pass

        except Exception as e:
            self.running = False

            try:
                self.result_queue.put_nowait({
                    "error": str(e)
                })
            except queue.Full:
                pass

    def estimate_relative_distance(self, strength):
        """
        將聲音強度轉成 0~1 的相對距離：
          0 = 非常近 / 中心
          1 = 遠 / 外圈

        使用對數尺度，因為音量與感知距離不是線性的。
        這不是公尺數，只是遊戲內的相對遠近。
        """
        near_ref = max(float(self.near_ref_var.get()), 1e-6)
        far_ref = max(float(self.far_ref_var.get()), 1e-6)

        if near_ref <= far_ref:
            near_ref = far_ref * 1.5

        s = max(float(strength), 1e-8)

        ln_near = math.log(near_ref)
        ln_far = math.log(far_ref)
        ln_s = math.log(s)

        d = (ln_near - ln_s) / (ln_near - ln_far)
        return float(np.clip(d, 0.0, 1.0))

    def process_results(self):
        try:
            while True:
                result = self.result_queue.get_nowait()

                if "error" in result:
                    self.status_var.set("擷取失敗")
                    messagebox.showerror(
                        "Audio Error",
                        result["error"]
                    )
                    continue

                self.current_result = result

                if not result.get("active", False):
                    continue

                raw_angle = result["angle"]

                self.angle_smoothed = circular_smooth(
                    self.angle_smoothed,
                    raw_angle
                )

                self.angle_var.set(
                    f"{self.angle_smoothed:+.1f}°"
                )

                self.conf_var.set(
                    f"{result['confidence']:.0f}%"
                )

                self.score_var.set(
                    f"{result['best_score']:.3f}"
                )

                self.second_var.set(
                    f"{result['second_angle']:+.0f}° / "
                    f"{result['second_score']:.3f}"
                )

                self.strength_var.set(
                    f"{result['strength']:.4f}"
                )

                raw_distance = self.estimate_relative_distance(
                    result['strength']
                )
                self.distance_smoothed = (
                    0.82 * self.distance_smoothed
                    + 0.18 * raw_distance
                )
                result['distance_norm'] = self.distance_smoothed

                if self.distance_smoothed < 0.22:
                    distance_name = "VERY NEAR"
                elif self.distance_smoothed < 0.45:
                    distance_name = "NEAR"
                elif self.distance_smoothed < 0.72:
                    distance_name = "MID"
                else:
                    distance_name = "FAR"

                self.distance_var.set(distance_name)
                self.distance_pct_var.set(
                    f"{self.distance_smoothed * 100:.0f}%"
                )

                self.near_ref_label.config(
                    text=f"{self.near_ref_var.get():.3f}"
                )
                self.far_ref_label.config(
                    text=f"{self.far_ref_var.get():.3f}"
                )

                lines = []

                for item in result["top"]:
                    lines.append(
                        f"{item['angle']:+7.1f}°   "
                        f"{item['score']:.3f}"
                    )

                self.top_var.set("\n".join(lines))

        except queue.Empty:
            pass

        self.draw_radar()
        self.root.after(30, self.process_results)

    def draw_radar(self):
        c = self.canvas
        c.delete("all")

        w = max(c.winfo_width(), 600)
        h = max(c.winfo_height(), 580)

        cx = w / 2
        cy = h / 2
        radius = min(w, h) * 0.38

        for frac in (1.0, 0.66, 0.33):
            r = radius * frac

            c.create_oval(
                cx-r, cy-r,
                cx+r, cy+r,
                outline="#555555"
            )

        # 距離圈：中心近、外圈遠
        c.create_text(cx + radius * 0.33, cy - 10, text="NEAR", fill="#777777")
        c.create_text(cx + radius * 0.66, cy - 10, text="MID", fill="#777777")
        c.create_text(cx + radius * 0.98, cy - 10, text="FAR", fill="#777777")

        c.create_line(
            cx-radius, cy,
            cx+radius, cy,
            fill="#555555"
        )

        c.create_line(
            cx, cy-radius,
            cx, cy+radius,
            fill="#555555"
        )

        c.create_text(
            cx, cy-radius-24,
            text="FRONT 0°",
            fill="white"
        )

        c.create_text(
            cx+radius+42, cy,
            text="RIGHT +90°",
            fill="white"
        )

        c.create_text(
            cx-radius-42, cy,
            text="LEFT -90°",
            fill="white"
        )

        c.create_text(
            cx, cy+radius+24,
            text="BACK ±180°",
            fill="white"
        )

        c.create_oval(
            cx-8, cy-8,
            cx+8, cy+8,
            fill="white",
            outline=""
        )

        c.create_text(
            cx, cy+23,
            text="PLAYER",
            fill="white"
        )

        result = self.current_result

        if (
            result is None
            or not result.get("active", False)
        ):
            c.create_text(
                cx, 30,
                text="等待聲音...",
                fill="#aaaaaa",
                font=("Arial", 16)
            )
            return

        angle = self.angle_smoothed
        theta = math.radians(angle)

        # 距離決定半徑：
        # 0 = 中心（很近），1 = 外圈（很遠）
        distance_norm = float(result.get("distance_norm", 0.5))

        # 保留極小半徑，避免標籤完全壓在 PLAYER 上；
        # 當距離接近 0 時仍會非常靠近中心。
        rr = radius * (0.04 + 0.86 * distance_norm)

        x = cx + rr * math.sin(theta)
        y = cy - rr * math.cos(theta)

        confidence = result["confidence"]

        c.create_line(
            cx, cy,
            x, y,
            fill="#dddddd",
            width=2
        )

        c.create_oval(
            x-13, y-13,
            x+13, y+13,
            fill="white",
            outline=""
        )

        c.create_text(
            x,
            y-34,
            text=(
                f"{angle:+.1f}°\n"
                f"DIST {distance_norm * 100:.0f}%  CONF {confidence:.0f}%"
            ),
            fill="white",
            font=("Arial", 12, "bold"),
            justify="center"
        )

        # 低信心提示
        if confidence < 30:
            c.create_text(
                cx,
                32,
                text="前後特徵不足 / 多聲源混合，方向可能不可靠",
                fill="#aaaaaa",
                font=("Arial", 13)
            )

    def on_close(self):
        self.running = False
        self.root.destroy()


if __name__ == "__main__":
    root = tk.Tk()
    app = AudioRadarV4(root)
    root.mainloop()